## American Community Survey, 5-Year Estimate Detailed Tables

The entry-point for the `acspsuedo` package is the `data` module, particularly the `download_by_geo_collection()` function.

In [1]:
import acspsuedo.data as apd

### Asynchronous GET Protocols

`acspsuedo` makes use of asynchronous GET protocols to the Census Bureau's database.

However, these protocols are decorated with a runtime decorator to execute the coroutine as a normal synchronous function if a running event loop is not found.

Thus, in a Jupyter Notebook environment (such as this present), the returned object would not be a `pandas.DataFrame` (or `geopandas.GeoDataFrame`, if geometric information is specified) but an awaitable.

In [2]:
from acspsuedo.api import ACS5
from acspsuedo.fips.states import CA
from acspsuedo.fips.places.CA import LOS_ANGELES

df = apd.download_by_geo_collection(
    api  = ACS5,
    year = 2023,
    dataset = 'B25058',
    geography = 'tract',
    state = CA,
    place = LOS_ANGELES
)

df

<coroutine object download_by_geo_collection at 0x11a5fbc40>

Awaiting the result would return the desired `pandas.DataFrame` object.

In [3]:
await df

,NAME,ucgid,GEOID,B25058_001E,B25058_001M
0,Census Tract 1011.10; Los Angeles County; Cali...,1400000US06037101110,06037101110,1486,309
1,Census Tract 1011.22; Los Angeles County; Cali...,1400000US06037101122,06037101122,3291,854
2,Census Tract 1012.20; Los Angeles County; Cali...,1400000US06037101220,06037101220,1298,132
3,Census Tract 1012.21; Los Angeles County; Cali...,1400000US06037101221,06037101221,1666,109
4,Census Tract 1012.22; Los Angeles County; Cali...,1400000US06037101222,06037101222,1539,315
...,...,...,...,...,...
1116,Census Tract 9800.26; Los Angeles County; Cali...,1400000US06037980026,06037980026,-666666666,-222222222
1117,Census Tract 9800.28; Los Angeles County; Cali...,1400000US06037980028,06037980028,-666666666,-222222222
1118,Census Tract 9800.31; Los Angeles County; Cali...,1400000US06037980031,06037980031,-666666666,-222222222
1119,Census Tract 9800.39; Los Angeles County; Cali...,1400000US06037980039,06037980039,-666666666,-222222222


Note too that Federal Information Processing Standard (FIPS) codes are provided for select upper-level/parent geographic scopes. Some upper-level scopes may required complete specification (e.g. `county` implicitly implies `state`).

### Adding Geometric Information

Specifying geometries would return a `geopandas.GeoDataFrame` object.

In [4]:
gdf = await apd.download_by_geo_collection(
    api  = ACS5,
    year = 2023,
    dataset = 'B25058',
    geography = 'tract',
    include_geometries = True,
    state = CA,
    place = LOS_ANGELES
)

gdf.head(3)

,NAME,ucgid,GEOID,B25058_001E,B25058_001M,STATEFP,COUNTYFP,TRACTCE,INTPTLAT,INTPTLON,geometry
0,Census Tract 1011.10; Los Angeles County; Cali...,1400000US06037101110,06037101110,1486,309,06,037,101110,34.259474,-118.292987,"POLYGON ((-118.30229 34.2587, -118.30091 34.25..."
1,Census Tract 1011.22; Los Angeles County; Cali...,1400000US06037101122,06037101122,3291,854,06,037,101122,34.267721,-118.290147,"POLYGON ((-118.30334 34.27371, -118.3033 34.27..."
2,Census Tract 1012.20; Los Angeles County; Cali...,1400000US06037101220,06037101220,1298,132,06,037,101220,34.251608,-118.281633,"POLYGON ((-118.28592 34.25227, -118.28592 34.2..."


### Caching Behavior

Note that to handle the query of large byte TIGER shapefiles, local caching is the default behavior. Thus, any extracted shapefiles, as well as the API key (if you choose to supply it), are stored in a created `cache/` folder of the working directory.

View and manage caching preferences with the `CONFIG` interface.

In [5]:
apd.CONFIG.is_cache # Default behavior is 'None', which is resolved as True

True

In [6]:
apd.CONFIG.api_key # Unsupplied, so an empty string is returned

''

Setting a new cache path will move all files (and directories) from the old cache path into the new one. The old cache path is then deleted.

In [7]:
apd.CONFIG.cache_path = 'new_cache' # Is the new cache path 'new_cache'? Check it for yourself!

print(apd.CONFIG.cache_path)

new_cache
